In [23]:
%load_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
from datetime import timedelta

from analyse.advertising.e00_download.mediatree import (
    CachedMediatreeAPI,
    all_intervals_between,
)
from analyse.advertising.e02_finger_printer.finger_printer import (
    FingerprintPipeline,
    print_report,
    plot_groups,
)

from datetime import datetime
from zoneinfo import ZoneInfo
import json
from pathlib import Path

tz_paris = ZoneInfo("Europe/Paris")
channel = "tf1"
output_prefix = "INVERSED"
TESTIMONY_CHANNEL = "TF1"

MONDAY_LAST_YEAR = datetime(2025, 3, 11, tzinfo=tz_paris)

SIMILARITY_THRESHOLD = 0.2

OUTPUT_FILE = f"{output_prefix}_report_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.json"
OUTPUT_PLOT = f"{output_prefix}_plot_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.png"

INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)


In [25]:
api = CachedMediatreeAPI()

INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)

input_files = []

intervals = all_intervals_between(
    MONDAY_LAST_YEAR, MONDAY_LAST_YEAR + timedelta(days=7), timedelta(minutes=30)
)

intervals = intervals[24:28]


In [26]:
mediatree_api = CachedMediatreeAPI()
import httpx
async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client:
    witness_files = [
        await mediatree_api.api.get_single_export_url(
            client,
            channel, start_time, end_time + timedelta(minutes=1), media_format="mp4"
        )
        for start_time, end_time in intervals
    ]


In [27]:
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)
from analyse.advertising.tools.basic_elements.segments import load_segment_groups_from_json
from analyse.advertising.tools.testimony_table.extract import get_testimony_data


for witness_file, (start_time, end_time) in zip(witness_files, intervals):
    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    segments = []  # load_segment_groups_from_json(segment_file)

    annotations = get_testimony_data(
        channel=TESTIMONY_CHANNEL,
        from_date=start_time,
        to_date=end_time,
        source_file="export.csv",
    )
    generate_player(
        media_input=witness_file,
        segments=[s.to_dict() for s in segments],
        annotations=format_annotation(annotations, from_date=start_time),
        output_path=f"output_{start_time.strftime('%Y-%m-%d_%H-%M-%S')}.html",
        # start_epoch=int(start_time.timestamp()),
    )


[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-11T11-00-00Z_2025-03-11T11-31-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  0 segments
[3/3] Annotations fournies en entrée
  1 annotations

Génération du rapport HTML : output_2025-03-11_12-00-00.html...

✓  Rapport généré  : output_2025-03-11_12-00-00.html
   Taille HTML     : 0.1 Mo (léger, média chargé à la volée)
   URL média       : https://vod.mediatree.fr/assets/tf1_2025-03-11T11-00-00Z_2025-03-11T11-31-00Z.mp4
   Segments        : 0
   Annotations CSV : 1

   → Ouvrez dans Firefox ou Chrome pour l'analyse.

[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-11T11-30-00Z_2025-03-11T12-01-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  0 segments
[3/3] Annotations fournies en entrée
  3 annotations

Génération du rapport HTML : output_2025-03-11_12-30-00.html...

✓ 